# 04 · AI Agents

An **agent** is an LLM in a loop: **Reason** → **Act** (call a tool) → **Observe** (read the result) → repeat until done. This is **ReAct**.

| Paradigm | Flow |
|---|---|
| Direct prompting | Question → LLM → Answer |
| Tool-augmented | Question → LLM → Tool → LLM → Answer |
| **Agent** | Question → LLM → Tool → LLM → Tool → … → Answer |

### Choose your LLM backend

- `"ollama"`: a local model via [Ollama](https://ollama.com). Run `ollama pull gemma4:e2b-mlx` first.
- `"gemini"`: Google Gemini. Put `GEMINI_KEY=your-key` in a `.env` file in the repo folder ([get a key](https://aistudio.google.com/apikey)).
- `"api"`: any other OpenAI-compatible cloud API (OpenAI, Groq, OpenRouter, …). Set `LLM_API_KEY`, `LLM_API_MODEL` and, if not OpenAI, `LLM_API_BASE_URL`.

The rest of the notebook works the same with either backend.

In [1]:
# %pip install -q ollama openai numpy transformers torch ddgs python-dotenv
import importlib

import llm_client as llm

importlib.reload(llm)  # pick up edits to llm_client.py without restarting the kernel

BACKEND = "ollama"  # "ollama" (local), "gemini" (GEMINI_KEY in .env) or "api" (any OpenAI-compatible API)

llm.configure(BACKEND)
NATIVE_MODEL_ID = llm.current_model()
print(llm.describe())

Backend: Ollama (local)  |  model: gemma4:e2b-mlx


In [2]:
from typing import Any, Callable, Dict, List, Tuple

## Agent 1 · Thermostat

*"My teddy bear is cold. Please do something."* The agent checks the temperature, then sets the thermostat.

In [3]:
# ── A tiny simulated environment + two tools ─────────────────────────────────
HOUSE = {"living room": 62, "bedroom": 65}  # current temperatures in °F


def get_room_temperature(room: str) -> str:
    temp = HOUSE.get(room.lower())
    return f"The {room} is currently {temp}°F." if temp is not None else f"Unknown room: {room}. Known rooms: {list(HOUSE)}"


def set_thermostat(room: str, temperature_f: float) -> str:
    if room.lower() not in HOUSE:
        return f"Unknown room: {room}. Known rooms: {list(HOUSE)}"
    HOUSE[room.lower()] = temperature_f
    return f"Thermostat in the {room} set to {temperature_f}°F."


THERMOSTAT_TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "get_room_temperature",
            "description": "Read the current temperature (°F) of a room in the house",
            "parameters": {
                "type": "object",
                "properties": {"room": {"type": "string", "description": "Room name, e.g. 'living room' or 'bedroom'"}},
                "required": ["room"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "set_thermostat",
            "description": "Set the target temperature (°F) of a room's thermostat",
            "parameters": {
                "type": "object",
                "properties": {
                    "room": {"type": "string", "description": "Room name"},
                    "temperature_f": {"type": "number", "description": "Target temperature in °F"},
                },
                "required": ["room", "temperature_f"],
            },
        },
    },
]
THERMOSTAT_TOOL_REGISTRY: Dict[str, Callable[..., str]] = {
    "get_room_temperature": get_room_temperature,
    "set_thermostat": set_thermostat,
}

REACT_SYSTEM_PROMPT = (
    "You are a smart-home agent. The teddy bear lives in the living room. "
    "Work step by step: before each tool call, write one short sentence explaining your reasoning. "
    "Check the current state before changing anything. When you are done, tell the user what you did."
)

In [4]:
# ── The ReAct loop: Reason -> Act -> Observe, until the model stops calling tools ─
def run_react_agent(user_message: str, tools_schema: List[Dict], registry: Dict[str, Callable[..., str]],
                    system_prompt: str, max_steps: int = 6, model: str = NATIVE_MODEL_ID) -> str:
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_message}]
    for step in range(1, max_steps + 1):
        response = llm.chat(model=model, messages=messages, tools=tools_schema)
        messages.append(response.message)

        if response.message.content:
            print(f"[{step}] Thought:     {response.message.content.strip()}")
        if not response.message.tool_calls:
            return response.message.content  # no more actions — this is the final answer

        for tool_call in response.message.tool_calls:
            result = registry[tool_call.function.name](**tool_call.function.arguments)
            print(f"[{step}] Action:      {tool_call.function.name}({tool_call.function.arguments})")
            print(f"[{step}] Observation: {result}")
            messages.append({"role": "tool", "content": str(result), "tool_name": tool_call.function.name})

    return "Max steps reached without a final answer."


answer = run_react_agent(
    "My teddy bear is cold. Please do something.",
    THERMOSTAT_TOOLS_SCHEMA, THERMOSTAT_TOOL_REGISTRY, REACT_SYSTEM_PROMPT,
)
print(f"\nFinal answer: {answer}")
print("House state:", HOUSE)

[1] Action:      get_room_temperature({'room': 'living room'})
[1] Observation: The living room is currently 62°F.


[2] Thought:     The living room is currently 62°F, so I will set the thermostat to 72°F to warm up the area.
[2] Action:      set_thermostat({'room': 'living room', 'temperature_f': 72})
[2] Observation: Thermostat in the living room set to 72°F.


[3] Thought:     I have checked the living room temperature, found it was 62°F, and then set the thermostat to 72°F to warm up the area for your teddy bear.

Final answer: I have checked the living room temperature, found it was 62°F, and then set the thermostat to 72°F to warm up the area for your teddy bear.
House state: {'living room': 72, 'bedroom': 65}


**Try it:** *"Make every room 70°F"*.

## Agent 2 · Chaining tools

One question, two tools (weather + currency).

In [5]:
# ── Same two toy tools + JSON Schema definitions as notebook 03 ─────────────

def get_weather(city: str) -> str:
    """Toy weather lookup — a real implementation would call a weather API."""
    fake_weather = {"osaka": "28C, sunny", "tokyo": "26C, cloudy", "paris": "19C, rainy"}
    return fake_weather.get(city.lower(), f"No weather data for {city}")


def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Toy currency converter — a real implementation would call a live FX API."""
    fake_rates = {("USD", "JPY"): 147.5, ("JPY", "USD"): 1 / 147.5}
    rate = fake_rates.get((from_currency.upper(), to_currency.upper()))
    if rate is None:
        return f"No rate available for {from_currency} -> {to_currency}"
    return f"{amount * rate:.2f} {to_currency.upper()}"


# JSON Schema tool definitions — this is what the model actually sees
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a city",
            "parameters": {
                "type": "object",
                "properties": {"city": {"type": "string", "description": "City name"}},
                "required": ["city"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "convert_currency",
            "description": "Convert an amount of money from one currency to another",
            "parameters": {
                "type": "object",
                "properties": {
                    "amount": {"type": "number", "description": "Amount to convert"},
                    "from_currency": {"type": "string", "description": "3-letter source currency code"},
                    "to_currency": {"type": "string", "description": "3-letter target currency code"},
                },
                "required": ["amount", "from_currency", "to_currency"],
            },
        },
    },
]

NATIVE_TOOL_REGISTRY: Dict[str, Callable[..., str]] = {
    "get_weather": get_weather,
    "convert_currency": convert_currency,
}

In [6]:
# ── Live Test 3: multi-step agent loop (model may chain several tool calls) ──
def run_agent(user_message: str, max_steps: int = 4, model: str = NATIVE_MODEL_ID) -> str:
    messages = [{"role": "user", "content": user_message}]
    for step in range(max_steps):
        response = llm.chat(model=model, messages=messages, tools=TOOLS_SCHEMA)
        messages.append(response.message)
        print(f"\n[step {step + 1}] Model response: {response.message.content}")

        if not response.message.tool_calls:
            return response.message.content  # model is done — final answer

        for tool_call in response.message.tool_calls:
            fn = NATIVE_TOOL_REGISTRY[tool_call.function.name]
            result = fn(**tool_call.function.arguments)
            print(f"  [step {step + 1}] {tool_call.function.name}({tool_call.function.arguments}) -> {result}")
            messages.append({"role": "tool", "content": str(result), "tool_name": tool_call.function.name})
    return "Max steps reached without a final answer."

In [7]:
answer = run_agent("What's the weather in Tokyo, and how much is 50 USD in JPY?")
print(f"\nFinal answer: {answer}")


[step 1] Model response: 
  [step 1] get_weather({'city': 'Tokyo'}) -> 26C, cloudy
  [step 1] convert_currency({'amount': 50, 'from_currency': 'USD', 'to_currency': 'JPY'}) -> 7375.00 JPY



[step 2] Model response: The weather in Tokyo is 26°C and cloudy. 50 USD is equal to 7375.00 JPY.

Final answer: The weather in Tokyo is 26°C and cloudy. 50 USD is equal to 7375.00 JPY.


`run_agent` and `run_react_agent` are the same loop; only tools and prompt change.

## Agent 3 · Agentic RAG

Retrieval becomes a tool: the model decides *when* to search, then answers with `[n]` citations.

In [8]:
# The pet knowledge base and vector (embedding) retriever built step by step in notebook 02
from rag_utils import PET_CORPUS, VectorRetriever

pet_retriever = VectorRetriever(PET_CORPUS)
CORPUS_BY_ID = pet_retriever.by_id
vector_search = pet_retriever.vector_search

/Users/rinabuoy/miniforge3/envs/LLMs/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9910.61it/s]

In [9]:
# ── The retrieval tool itself, plus its JSON-Schema definition ────────────────
def retrieve_context(query: str, top_k: int = 3) -> str:
    """Tool: vector (embedding) search over the pet knowledge base; returns citation-ready text blocks."""
    results = vector_search(query, top_k=top_k)
    if not results:
        return "No results found."
    return "\n".join(f"[{i}] {CORPUS_BY_ID[doc_id]}" for i, (doc_id, _score) in enumerate(results, start=1))


RAG_TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "retrieve_context",
            "description": (
                "Vector (embedding) search over a small pet knowledge base. "
                "Returns numbered, citation-ready text blocks most relevant to the query."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Natural-language search query"},
                    "top_k": {"type": "integer", "description": "Number of results to return (default 3)"},
                },
                "required": ["query"],
            },
        },
    }
]

RAG_TOOL_REGISTRY: Dict[str, Callable[..., str]] = {"retrieve_context": retrieve_context}

RAG_SYSTEM_PROMPT = (
    "You are a helpful assistant answering questions about pets. Always call retrieve_context "
    "to find relevant information before answering. Cite sources using the [n] markers from the "
    "retrieved text. If the retrieved context doesn't contain the answer, say so."
)


async def run_rag_agent(user_message: str, max_steps: int = 3, model: str = NATIVE_MODEL_ID) -> str:
    messages = [
        {"role": "system", "content": RAG_SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]
    for step in range(max_steps):
        response = llm.chat(model=model, messages=messages, tools=RAG_TOOLS_SCHEMA)
        messages.append(response.message)

        if not response.message.tool_calls:
            return response.message.content  # model is done — final answer

        for tool_call in response.message.tool_calls:
            fn = RAG_TOOL_REGISTRY[tool_call.function.name]
            result = fn(**tool_call.function.arguments)
            print(f"  [step {step + 1}] {tool_call.function.name}({tool_call.function.arguments}) ->\n{result}")
            messages.append({"role": "tool", "content": result, "tool_name": tool_call.function.name})

    return "Max steps reached without a final answer."

In [10]:
# Needs facts from two different documents — the parrot (d4) and the Bengal cat (d5)
answer = await run_rag_agent(
    "Which pet can mimic human speech, and which pet is a hybrid with a wild leopard cat?"
)
print(f"\nFinal answer: {answer}")

  [step 1] retrieve_context({'query': 'pet that can mimic human speech'}) ->
[1] African Grey Parrots are known for their exceptional ability to mimic human speech and solve problems.
[2] Persian cats have long, thick fur and a calm, gentle temperament, making them popular indoor pets.
[3] Bearded dragons are docile reptiles native to Australia, popular as low-maintenance terrarium pets.
  [step 1] retrieve_context({'query': 'pet hybrid with a wild leopard cat'}) ->
[1] Bengal cats are a hybrid breed resulting from crossing domestic cats with the Asian leopard cat.
[2] Persian cats have long, thick fur and a calm, gentle temperament, making them popular indoor pets.
[3] Bearded dragons are docile reptiles native to Australia, popular as low-maintenance terrarium pets.



Final answer: Based on the retrieved information:

*   The pet that can mimic human speech is the **African Grey Parrot** [1].
*   The pet that is a hybrid with a wild leopard cat is the **Bengal cat**, which is a hybrid breed resulting from crossing domestic cats with the Asian leopard cat [1].


## Agent 4 · Web search

Live web search via [`ddgs`](https://pypi.org/project/ddgs/) (DuckDuckGo, no API key). Fresher than RAG, but less controlled and less reliable.

In [11]:
# %pip install -q ddgs

In [12]:
# ── The tool itself, plus its JSON-Schema definition ──────────────────────────
from ddgs import DDGS


def web_search(query: str, max_results: int = 5) -> str:
    """Tool: search the live web via DuckDuckGo; returns numbered, citation-ready text blocks."""
    results = DDGS().text(query, max_results=max_results)
    if not results:
        return "No results found."
    return "\n".join(
        f"[{i}] {r['title']}\n    {r['href']}\n    {r['body']}" for i, r in enumerate(results, start=1)
    )


WEB_TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": (
                "Search the live web via DuckDuckGo. Returns numbered, citation-ready text "
                "blocks (title, URL, snippet) for the top results."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query"},
                    "max_results": {"type": "integer", "description": "Number of results to return (default 5)"},
                },
                "required": ["query"],
            },
        },
    }
]

WEB_TOOL_REGISTRY: Dict[str, Callable[..., str]] = {"web_search": web_search}

# Sanity check: call the tool directly, no model involved yet
print(web_search("Ollama latest release version", max_results=3))

[1] Releases · ollama/ollama
    https://github.com/ollama/ollama/releases
    3 weeks ago - Ollama models can now be used directly in ChatGPT Desktop, so you can keep your existing workflow while running open models. Setup is available from the Ollama app on MacOS. This release also improves structured output performance on Apple Silicon, adds support for OpenAI-compatible client tool search and response compaction.
[2] Ollama Release Notes - September 2026 Latest Updates - Releasebot
    https://releasebot.io/updates/ollama
    5 days ago - Ollama updates MLX with ModelOpt ... and a version bump. ... Ollama adds ChatGPT Desktop support, faster structured output on Apple Silicon, and new OpenAI-compatible tool search and response compaction. Ollama models can now be used directly in ChatGPT Desktop, so you can keep your existing workflow while running open models. Setup is available from the Ollama app on MacOS. This release also improves ...
[3] Ollama Latest Version 2026: v0.33.1 + 

### Wiring it into the loop

Same loop, new tool. The question needs up-to-date information.

In [13]:
WEB_SYSTEM_PROMPT = (
    "You are a helpful assistant with access to a live web search tool. Call web_search "
    "whenever the answer depends on current or fast-changing information you might not know. "
    "Cite sources using the [n] markers from the search results."
)


async def run_web_agent(user_message: str, max_steps: int = 3, model: str = NATIVE_MODEL_ID) -> str:
    messages = [
        {"role": "system", "content": WEB_SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]
    for step in range(max_steps):
        response = llm.chat(model=model, messages=messages, tools=WEB_TOOLS_SCHEMA)
        messages.append(response.message)

        if not response.message.tool_calls:
            return response.message.content  # model is done — final answer

        for tool_call in response.message.tool_calls:
            fn = WEB_TOOL_REGISTRY[tool_call.function.name]
            result = fn(**tool_call.function.arguments)
            print(f"  [step {step + 1}] {tool_call.function.name}({tool_call.function.arguments}) ->\n{result}")
            messages.append({"role": "tool", "content": result, "tool_name": tool_call.function.name})

    return "Max steps reached without a final answer."

In [14]:
answer = await run_web_agent("What is the most recent version of Ollama, and when was it released?")
print(f"\nFinal answer: {answer}")

  [step 1] web_search({'query': 'most recent version of Ollama and release date'}) ->
[1] Ollama - Wikipedia
    https://en.wikipedia.org/wiki/Ollama
    1 day ago - In January 2026, The Hacker News ... despite the fact that Ollama is intended to run locally by default. ... ↑ SitePoint Team (April 22, 2026). "10GB VRAM Local LLM: The Complete Setup Guide (2026)". SitePoint. Retrieved April 29, 2026. ↑ "Release v0.0.1"....
[2] Ollama Release Notes - September 2026 Latest Updates - Releasebot
    https://releasebot.io/updates/ollama
    5 days ago - Ollama updates MLX with ModelOpt global scale support for MoE models and a version bump.
[3] Ollama Latest Version 2026: v0.33.1 + 10 Best Models
    https://www.promptquorum.com/local-llms/top-open-source-models-ollama
    1 month ago - The current Ollama version is v0.33.1 (August 26, 2026); v0.33.2 is available as a release candidate. The newest models added recently are Laguna XS 2.1 (Poolside, 33B/3B active MoE, agentic coding) and Kimi 


Final answer: Based on the search results, the most recent version of Ollama mentioned is **v0.33.2**.

This version was released on **August 27, 2026** [5]. The search also notes that version v0.33.1 was released on August 26, 2026 [3].


## Key takeaways

- Agent = LLM + tools + a loop (Reason → Act → Observe).
- Tools and system prompt define the agent.
- Always cap the loop with `max_steps`.